# Q1. 1293. Gradient clipping




Scale all gradients down by a shared factor only when their combined L2 norm exceeds `max_norm`.

$$\text{global\_norm} = \sqrt{\sum_i \|g_i\|_2^2}, \qquad \text{scale} = \min\!\left(1,\ \frac{\text{max\_norm}}{\text{global\_norm}}\right), \qquad g_i' = \text{scale} \cdot g_i$$

**Constraints**
- If `global_norm <= max_norm` → return gradients unchanged
- Single shared scale applied to all gradient vectors

**Input**
```
grads = [
    np.array([3.0,  4.0]),   # norm = 5
    np.array([0.0, 12.0])    # norm = 12
]
max_norm = 10.0
```
**Expected Output**
```
[array([2.3077, 3.0769]),
 array([0.0,    9.2308])]

global_norm = sqrt(3² + 4² + 0² + 12²) = sqrt(169) = 13
scale       = 10 / 13 ≈ 0.7692

[3.0,  4.0] × 0.7692 = [2.3077, 3.0769]
[0.0, 12.0] × 0.7692 = [0.0,    9.2308]
```

In [ ]:
import numpy as np

def clip_gradients(grads, max_norm):
    """
    Applies global norm gradient clipping.

    Args:
        grads (list[np.ndarray]): A list of gradient vectors (each is a 1D
            np.ndarray).
        max_norm (float): The maximum allowed global L2 norm.

    Returns:
        list[np.ndarray]: Clipped gradients, same nested-list structure as
            `grads`.
    """
    # Ensure inputs are arrays (in case lists are passed via tests)
    grads = [np.asarray(g, dtype=float) for g in grads]

    # Compute the global squared L2 norm
    total_sq = sum(np.sum(g**2) for g in grads)

    # Compute global L2 norm
    global_norm = np.sqrt(total_sq)

    # If within the limit, return a copy with same values
    if global_norm <= max_norm:
        return [g.copy() for g in grads]

    # Compute scaling factor
    scale = max_norm / global_norm

    # Scale all gradients
    return [g * scale for g in grads]

grads = [
    np.array([3.0,  4.0]),   # norm = 5
    np.array([0.0, 12.0])    # norm = 12
]
max_norm = 10.0

result = clip_gradients(grads, max_norm)
result

array([[2.30769231, 3.07692308],
       [0.        , 9.23076923]])

# Q2. 177. Least squares 1D closed form




Fit a line $y \approx mx + b$ by minimizing sum of squared errors. Set partial derivatives to zero and solve directly — no gradient descent.

$$\min_{m,b} \sum_{i=1}^{n}(y_i - (mx_i + b))^2$$

$$m = \frac{n\sum x_i y_i - \sum x_i \sum y_i}{n \sum x_i^2 - \left(\sum x_i\right)^2}, \qquad b = \frac{\sum y_i - m\sum x_i}{n}$$

**Input**
```
x = [1.0, 2.0, 3.0]
y = [2.0, 3.0, 5.0]
```
**Expected Output**
```
(1.5, 0.333)

n=3, Σx=6, Σy=10, Σx²=14, Σxy=23

m = (3×23 − 6×10) / (3×14 − 6²) = (69−60) / (42−36) = 9/6 = 1.5
b = (10 − 1.5×6) / 3 = 1/3 ≈ 0.333
```

In [6]:
import numpy as np

def least_squares_1d(x, y):
    """
    Computes the closed-form least squares fit for y ≈ m*x + b.

    Args:
        x (np.ndarray): 1D input values of length n.
        y (np.ndarray): 1D target values of length n.

    Returns:
        tuple[float, float]: (m, b) where m is slope and b is intercept.
    """
    n = len(x)

    # Compute sufficient statistics using vectorized sums
    sum_x = np.sum(x)
    sum_y = np.sum(y)
    sum_xx = np.sum(x * x)
    sum_xy = np.sum(x * y)

    # Closed-form slope and intercept
    denom = n * sum_xx - sum_x * sum_x
    # Optionally handle division by zero or assume valid input per problem constraints
    m = (n * sum_xy - sum_x * sum_y) / denom
    b = (sum_y - m * sum_x) / n

    return m, b

x = np.array([1.0, 2.0, 3.0])
y = np.array([2.0, 3.0, 5.0])

result = least_squares_1d(x, y)
result

(np.float64(1.5), np.float64(0.3333333333333333))

# Q3. 832. Minimize 1D quadratic Closed Form




Find the $x$ that minimizes a 1D quadratic. Set the derivative to zero and solve directly.

$$f(x) = ax^2 + bx + c$$

$$f'(x) = 2ax + b = 0 \implies x^* = -\frac{b}{2a}$$

**Input**
```
a, b, c = 2.0, -8.0, 3.0
```
**Expected Output**
```
2.0

x* = -(-8.0) / (2 × 2.0) = 8 / 4 = 2.0
```

In [ ]:
import numpy as np

def minimize_quadratic_1d(a, b, c):
    """
    Finds the minimizer x* of a 1D quadratic function.

    The objective is:
        f(x) = a * x^2 + b * x + c

    Args:
        a (float): Quadratic coefficient.
        b (float): Linear coefficient.
        c (float): Constant term.

    Returns:
        float: The value x* that minimizes f(x).
    """
    # Set derivative 2 * a * x + b to zero and solve for x
    x_star = -b / (2.0 * a)
    return x_star

result = minimize_quadratic_1d(a, b, c)
result

# Q4. 625. Projection onto box constraints




Clip each element independently into its allowed interval $[\ell_i, u_i]$.

$$\Pi_{[\ell,u]}(x)_i = \min\!\left(\max(x_i,\ \ell_i),\ u_i\right)$$

**Input**
```
x     = [ 2.5, -1.0, 0.3, 10.0]
lower = [ 0.0,  0.0, 0.0,  3.0]
upper = [ 2.0,  5.0, 0.5,  7.0]
```
**Expected Output**
```
[2.0, 0.0, 0.3, 7.0]

 2.5  → above upper → clipped to 2.0
-1.0  → below lower → clipped to 0.0
 0.3  → in range    → unchanged  0.3
10.0  → above upper → clipped to 7.0
```

In [ ]:
import numpy as np

def project_onto_box(x, lower, upper):
    """
    Projects a vector onto box constraints (elementwise bounds).

    Args:
        x (np.ndarray): Input vector of length n.
        lower (np.ndarray): Lower bounds of length n.
        upper (np.ndarray): Upper bounds of length n.

    Returns:
        np.ndarray: Projected vector y where each y[i] is in
            [lower[i], upper[i]].
    """
    # Clip each element to its [lower, upper] interval
    return np.clip(x, lower, upper)

result = project_onto_box(x, lower, upper)
result